# Week 9 — Finish the retreat (target 50/50)

## Where week 8b left off
The 3,000-neuron readout completes **25/50** (week 8: 0/50 with the DNs). Every episode lifts the cube and carries it to the zone. **All 25 failures are the same thing:** the cube is put down and released, then the decoded dz sits at about 0 ± 10 mm/step and the open gripper hovers just above the cube until time runs out (success needs it 6 cm clear).

**Why:** a cliff in the teacher's label. While the fingers open (3 steps, width 50 → 67 mm) the expert says dz = 0; one step later, past 70 mm, it says +20 mm/step. A linear readout of noisy rates averages neighbouring states and predicts ≈ 0.

## The candidates (all keep the frozen connectome and a linear readout)
| | change | what it targets |
|---|---|---|
| **C0** | none (week 8b's recipe) | baseline |
| **C1** | re-choose the 3,000 neurons at every refit on **all** DAgger data | round 0 has ~4 retreat steps per episode; the brain gets stuck in states only later rounds contain |
| **C2** | **weight** the release region (cube on the table at the target) to ≥ 1/3 of the fit | the rare end phase |
| **C3** | **B2 graded retreat** (`expert9.py`): dz ramps 0 → +20 as the fingers open | the cliff in the label |
| **C4** | C1 + C2 | |
| **C5** | C1 + C2 + C3 | |
| G | two motion readouts switched by the decoded grip | **report only, never chosen** (not linear): tells week 10 what a slightly nonlinear readout adds |

**B2 checked without a brain (sandbox, same physics as Kaggle):** expert 50/50, 2.7 mm, 1.88 s (fix B: 50/50, 2.7 mm, 1.92 s). Linear on the 8 signals 0/50, MLP 50/50, so the task still needs the nonlinear processing (week 8's control).

## This notebook
1. **Offline, no brain time (~15 min CPU):** week 8b's saved DAgger data (rates of all ~9,500 recorded neurons at every visited state, rounds 0–3) refitted with each candidate, out of episode (5 folds by whole episode). Scored on:
   - **trap: dz up** — released, fingers open past 70 mm, gripper not yet 6 cm clear: share of steps with decoded dz > +3 mm/step;
   - **press: dz down** — holding the cube within 4 cm above the release height near the target: share with dz < −3 mm/step;
   - dx, dy, dz R², grip balanced accuracy on all rows.
2. **The rule (fixed before the run):** end score = min(trap up, press down). Guards vs C0: dx and dy R² no more than 0.05 lower, grip no more than 0.02 lower.
   - **closed loop** with the best candidate whose end score ≥ 0.85 (a simpler one within 0.03 wins);
   - **partial:** none reaches 0.85, but the best beats C0's end score by ≥ 0.10 → run it anyway;
   - **stop:** otherwise. Week 10 is then about the readout itself.
3. **Closed loop (~45 min T4):** week 8b's pipeline (fix B loop, 80/80 timing, DAgger 4 × 30, the same seeds and test) with the chosen candidate. **Pass ≥ 40/50 (unchanged); target 50/50.**

## Kaggle inputs
`arm-sim-code` (updated with `sim/expert9.py`, `sim/week9.py`), `arm-sim-data`, and **week 8b's output**: Add Input → Your Work → the week 8b notebook (it holds `week8b/week8_connectome__wide_readout_b_dagger.npz`, ~0.5 GB). The setup cell links it into `data/week8b/`.

### Setup (the shared Kaggle / Colab / laptop cell)

In [ ]:
import sys, subprocess, os, shutil, zipfile, re
from pathlib import Path

# ---- where are we? -------------------------------------------------------------------------------
# Kaggle is checked first: its image also ships the google.colab package, but can't mount Drive.
KAGGLE_INPUT = Path('/kaggle/input')
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or KAGGLE_INPUT.is_dir()
IN_COLAB = not IN_KAGGLE and Path('/var/colab/hostname').exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
HOST = 'colab' if IN_COLAB else 'kaggle' if IN_KAGGLE else 'laptop'
if IN_COLAB or IN_KAGGLE:
    os.environ.setdefault('MUJOCO_GL', 'egl')                      # headless video; set before mujoco loads
    os.environ.setdefault('NUMBA_CACHE_DIR', '/tmp/numba_cache')   # the code folder is read-only on Kaggle

# ---- packages ------------------------------------------------------------------------------------
PKGS = ['brian2', 'numba', 'mujoco', 'imageio', 'imageio-ffmpeg']
if IN_COLAB or IN_KAGGLE:
    try:
        import brian2, numba, mujoco, imageio_ffmpeg
    except ImportError:
        if IN_KAGGLE:
            # A batch run (Save & Run All) can't survive a kernel restart, so numpy is pinned to the
            # installed version: nothing gets upgraded, nothing needs a restart.
            import numpy
            r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PKGS, f'numpy=={numpy.__version__}'])
            if r.returncode:
                raise RuntimeError("pip install failed. Notebook settings (right panel) -> Internet: On. "
                                   "Kaggle only offers Internet to phone-verified accounts.")
            import brian2, numba, mujoco, imageio_ffmpeg
        else:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PKGS], check=True)
            print("packages installed -> restarting runtime. When it reconnects, choose Runtime -> Run all again.")
            os.kill(os.getpid(), 9)

# ---- folders -------------------------------------------------------------------------------------
# Kaggle: the inputs are read-only and can come from several datasets and earlier notebook versions.
# They are linked into one tree with the Drive layout (arm_simulation/brain/{sim,data}, arm/,
# mujoco_menagerie/), so every path below is the same as on Colab. Results go to /kaggle/working.
def _make_real(d):
    """Make folder d (under ROOT) a real folder. Any part of its path that is a link into a read-only input
    becomes a real folder holding links to that input's contents, so a second input can add to it."""
    p = ROOT
    p.mkdir(parents=True, exist_ok=True)
    for name in Path(d).relative_to(ROOT).parts:
        p = p / name
        if p.is_symlink() and p.is_dir():
            t = p.resolve(); p.unlink(); p.mkdir()
            for k in sorted(t.iterdir()):
                (p / k.name).symlink_to(k)
        elif not p.exists():
            p.mkdir()

def _link_tree(src, dst):
    """Link everything under src into dst, folder by folder, so several inputs can fill one tree."""
    _make_real(dst)
    for s in sorted(src.iterdir()):
        d = dst / s.name
        if d.is_symlink() and s.is_dir() and d.is_dir():   # a folder another input already linked: split it
            t = d.resolve(); d.unlink(); _link_tree(t, d)
        if s.is_dir() and d.is_dir() and not d.is_symlink():
            _link_tree(s, d)
        elif not d.exists() and not d.is_symlink():
            d.symlink_to(s)

_RUN = re.compile(r'week\d+[a-z]*(_[a-z]+)?')        # week7, week8b, week8b_quick: a notebook's results folder

def _find_roots(top, depth=4):
    """('tree', p): p holds brain/, arm/ or mujoco_menagerie/ (an uploaded dataset).
       ('data', p): p is brain/data/ itself (weights_*/, wsyn_choice.json).
       ('runs', p): p holds week*/ folders (an earlier notebook version's output)."""
    found = []
    def walk(p, d):
        try:
            kids = sorted(p.iterdir())
        except OSError:
            return
        names = {k.name for k in kids}
        zips = [z for z in kids if z.suffix == '.zip' and z.stem in ('brain', 'arm', 'mujoco_menagerie')
                and z.stem not in names]           # in case an upload kept its folders zipped
        if zips:
            out = Path('/tmp/kaggle_unzip') / p.name
            for z in zips:
                if not (out / z.stem).exists():
                    zf = zipfile.ZipFile(z)
                    inside = all(n.startswith(z.stem + '/') for n in zf.namelist())
                    zf.extractall(out if inside else out / z.stem)
            found.append(('tree', out))
        if d > 0 and names & {'brain', 'arm', 'mujoco_menagerie'}:
            found.append(('tree', p))
        elif d > 0 and ('wsyn_choice.json' in names or any(n.startswith('weights_') for n in names)):
            found.append(('data', p))              # brain/data itself (an upload can drop the brain/ level)
        elif d > 0 and any(k.is_dir() and _RUN.fullmatch(k.name) for k in kids):
            found.append(('runs', p))
        elif d < depth:
            for k in kids:
                if k.is_dir():
                    walk(k, d + 1)
    walk(top, 0)
    return found

if IN_KAGGLE:
    ROOT = Path('/tmp/arm_simulation')                     # links only, rebuilt every session
    shutil.rmtree(ROOT, ignore_errors=True)
    ROOTS = _find_roots(KAGGLE_INPUT)
    for kind, p in sorted(ROOTS, key=lambda r: r[0] != 'runs'):   # notebook outputs first: newest results win
        if kind == 'tree':
            _link_tree(p, ROOT)
        elif kind == 'data':
            _link_tree(p, ROOT / 'brain' / 'data')
        else:
            for k in sorted(p.iterdir()):
                if k.is_dir() and _RUN.fullmatch(k.name):
                    _link_tree(k, ROOT / 'brain' / 'data' / k.name)
    print("Kaggle inputs:", *[f"{k:5s} {p}" for k, p in ROOTS], sep="\n  ")
    BRAIN = ROOT / 'brain'
elif IN_COLAB:
    BRAIN = Path('/content/drive/MyDrive/arm_simulation/brain')
else:
    BRAIN = Path.home() / 'arm_simulation/brain'
DATA = BRAIN / 'data'
RUNS = Path('/kaggle/working') if IN_KAGGLE else DATA      # where this notebook's results go
ARM = next((p for p in [BRAIN.parent / 'arm', BRAIN.parent / 'week4/arm'] if (p / 'panda_env.py').exists()), None)
sys.path.insert(0, str(BRAIN))
if ARM: sys.path.insert(0, str(ARM))

def out_dir(name):
    """This notebook's results folder. On Kaggle it is /kaggle/working/<name>, saved with the version; if an
    earlier version's output is attached as an input, its <name>/ is copied in first, so RESUME carries on."""
    o = RUNS / name
    if RUNS != DATA and not o.exists() and (DATA / name).exists():
        shutil.copytree(DATA / name, o)
        print(f"resuming: copied the earlier run's {name}/ into {o}")
    o.mkdir(parents=True, exist_ok=True)
    return o

# ---- this notebook ----------------------------------------------------------------------------------
QUICK = False            # True: smoke test (offline on 24 episodes; closed loop 3 x 6 DAgger, 10 test seeds), in week9_quick/
RESUME = True
RUN_CLOSED_LOOP = 'auto' # 'auto': as the rule says; True: always (FORCE_CANDIDATE, else the rule's best); False: offline only
FORCE_CANDIDATE = None   # e.g. 'C5 B2 + reselect + weight' to override the rule (say so in the results)
RUN_REFERENCES = True    # expert / linear / MLP on the 8 signals with the chosen labels (CPU, ~3 min)
RUN_RANDOM = False       # after a pass: the random network with its own wide readout (~70 min more)
K_WIDE = 3000
USE_GPU = True
GPU_DTYPE = 'float32'
MAX_BATCH = 64

GPU = None
if USE_GPU:
    try:
        import torch
        if torch.cuda.is_available():
            GPU = f"{torch.cuda.get_device_name(0)} (x{torch.cuda.device_count()})"
            print(f"GPU: {GPU}; torch {torch.__version__}")
        else:
            print("USE_GPU = True but no GPU: Kaggle -> Session options -> Accelerator GPU T4 x2 (fine for the offline step only)")
    except ImportError:
        print("torch is not installed here -> numba on the CPU")

W8B = DATA / 'week8b'
for p in ['sim/batch7.py', 'sim/week7.py', 'sim/network7.py', 'sim/interface.py', 'sim/expert8.py', 'sim/expert9.py',
          'sim/week8.py', 'sim/pool8.py', 'sim/week9.py', 'sim/video7.py', 'data/weights_male-cns_v1.0/meta.json',
          'data/shortlist_week1.csv', 'data/wsyn_choice.json', 'data/week7/week7_random_network_meta.json',
          'data/week8/week8_connectome_b.pkl']:
    print(f"{p:55s} {'ok' if (BRAIN / p).exists() else 'MISSING'}")
CK8B = sorted(W8B.glob('week8_connectome*wide*_dagger.npz'))
RES8B = sorted(W8B.glob('week8_connectome*wide*_b.pkl'))
print(f"{'week 8b DAgger data':55s} {CK8B[0].name if CK8B else 'MISSING: attach the week 8b notebook output'}")
print(f"{'week 8b closed-loop result':55s} {RES8B[0].name if RES8B else 'MISSING'}")
print(f"{'week 4 arm code':55s} {ARM if ARM else 'MISSING'}")

### Load

In [ ]:
import json, time, gc, pickle
import numpy as np, pandas as pd
import warnings; warnings.filterwarnings('ignore', module='brian2')
from panda_env import PandaPickPlace, OBS_NAMES
from sim.week9 import (CANDIDATES, REPORT_ONLY, RULE, offline9, choose9, end_score, run_network9, run_references9,
                       EXPERTS)

T_START = time.perf_counter()
OUT = out_dir('week9_quick' if QUICK else 'week9')
env = PandaPickPlace()
N_ROUNDS = 3 if QUICK else 4
N_PER_ROUND = 6 if QUICK else 30
CFG = dict(
    train_seeds=[list(range(64_000 + 1000 * r, 64_000 + 1000 * r + N_PER_ROUND)) for r in range(N_ROUNDS)],  # = weeks 7-8b
    betas=[1.0] + [0.0] * (N_ROUNDS - 1),
    test_seeds=list(range(92_000, 92_000 + (10 if QUICK else 50))),                                         # = weeks 7-8b
    cut_seeds=list(range(92_000, 92_000 + (3 if QUICK else 10))),
    k=K_WIDE, n_video=2, use_gpu=GPU is not None, dtype=GPU_DTYPE, max_batch=MAX_BATCH, cpu_batch=1)
if not (CK8B and RES8B):
    raise FileNotFoundError("week 8b's output is not among the inputs. Kaggle: Add Input -> Your Work -> the week 8b "
                            "notebook (its output holds week8b/), then rerun from the setup cell. The setup cell's "
                            "'Kaggle inputs' list should show it as 'runs'.")
R8B = pickle.load(open(RES8B[0], 'rb'))
WIDE = np.asarray(R8B['record'])[np.asarray(R8B['pool_cols'])]     # week 8b's pool: active neurons + all DNs (global ids)
print(f"week 8b: {R8B['summary']['success']}/{R8B['summary']['n']}; pool {len(WIDE):,} neurons, "
      f"{len(R8B['record']):,} recorded; results in {OUT}")

## Step 1: offline, on week 8b's DAgger data (no brain time)

In [ ]:
D8 = dict(np.load(CK8B[0]))
POOL_COLS = np.asarray(R8B['pool_cols'])
if QUICK:                                                   # 6 episodes from each round
    keep = np.concatenate([np.unique(D8['g'][D8['round'] == r])[:6] for r in np.unique(D8['round'])])
    m = np.isin(D8['g'], keep); D8 = {k: v[m] for k, v in D8.items()}
print(f"{len(D8['g']):,} steps x {D8['X'].shape[1]:,} neurons, {len(np.unique(D8['g']))} episodes, "
      f"rounds {np.unique(D8['round']).tolist()}")
t0 = time.perf_counter()
ROWS, COUNTS = offline9(D8, POOL_COLS, k=K_WIDE)
OFF = pd.DataFrame(ROWS).set_index('candidate').drop(columns='s')
OFF['end score'] = [round(end_score(r), 3) for r in ROWS]
display(OFF)
OFF.to_csv(OUT / f'week9_offline_{HOST}.csv')
print(f"rows {COUNTS['rows']:,}; release region {COUNTS['region']:,} ({COUNTS['region_share']:.0%}); trap {COUNTS['trap']:,} "
      f"(by round {COUNTS['trap_by_round']}, labelled 'up' {COUNTS['trap_label_up']:.0%}); press {COUNTS['press']:,}; "
      f"release {COUNTS['release']:,}  [{(time.perf_counter() - t0) / 60:.0f} min]")
DECISION, PICK, WHY = choose9(ROWS)
print(f"\nRULE -> {DECISION}: {WHY}")
del D8; gc.collect()

## Step 2: closed loop with the chosen candidate (about 45 min on a T4)

Week 8b's pipeline unchanged except the candidate's settings. **Pass ≥ 40/50; target 50/50.** Also: the feedback cut (10 seeds; must fail), the trap statistics of any episode that put the cube down but didn't finish, and the no-brain references with the same labels.

In [ ]:
if RUN_CLOSED_LOOP == 'auto':
    RUN = DECISION in ('closed loop', 'partial')
else:
    RUN = bool(RUN_CLOSED_LOOP)
CHOSEN = FORCE_CANDIDATE or PICK or max((r for r in ROWS if r['candidate'] in CANDIDATES), key=end_score)['candidate']
if FORCE_CANDIDATE:
    print(f"FORCE_CANDIDATE overrides the rule ({DECISION}, {PICK}) -> {FORCE_CANDIDATE}")
NETS, REFS = {}, {}
if RUN:
    from sim.lif import WeightSet, peak_mem_gb
    ws = WeightSet(DATA / 'weights_male-cns_v1.0')
    W = json.loads((DATA / 'wsyn_choice.json').read_text())['w_scale']
    JO = ws.by_type(r'^JO-(C|E)', cls='cb_sensory')
    shortlist = pd.read_csv(DATA / 'shortlist_week1.csv')
    g2i = pd.Series(ws.neurons.index, index=ws.neurons.bodyId)
    DN_cand = g2i.loc[shortlist.query("group == 'middle_descending'").bodyId].to_numpy()
    ALL_DN = ws.idx(ws.neurons['cls'].astype(str).str.contains('descending', case=False))
    full = ws.subnetwork(np.arange(ws.N))
    ws._out = None; gc.collect()
    CAND = CANDIDATES.get(CHOSEN) or REPORT_ONLY[CHOSEN]
    assert not CAND.get('gated'), "the gated readout is report-only: no closed-loop policy for it in week 9"
    LABEL = f"connectome, {CHOSEN.split(' ')[0]}"
    NETS[LABEL] = run_network9(LABEL, full, JO, WIDE, env, OBS_NAMES, CFG, OUT, w_scale=W, cand=CAND,
                               resume=RESUME, extra_record=DN_cand)
    s = NETS[LABEL]['summary']
    PASS = s['success'] >= 0.8 * s['n']
    print(f"\n{LABEL.upper()} ({CHOSEN}): success {s['success']}/{s['n']} (95% CI {s['ci'][0]:.0%}-{s['ci'][1]:.0%}), "
          f"placement {s['place_mm_success']} mm, time {s['time_s']} s, {s['outcomes']}")
    print(f"PASS (>= 40/50): {'YES' if PASS else 'NO'};  TARGET 50/50: {'YES' if s['success'] == s['n'] else 'no'}")
    print("while collecting, by round:", [f"{r['success']:.0%}" for r in NETS[LABEL]['dagger']['log']])
    if NETS[LABEL].get('cut'):
        print("feedback cut:", NETS[LABEL]['cut']['summary']['outcomes'])
    if NETS[LABEL]['trap']:
        display(pd.DataFrame(NETS[LABEL]['trap']))
    if RUN_REFERENCES:
        REFS = run_references9(env, OBS_NAMES, CFG, CAND['labels'])
    if PASS and RUN_RANDOM:
        from sim.interface import Encoder
        from sim.batch7 import make_engine, BatchBrain
        from sim.network7 import wiring_labels, random_network
        from sim.week7 import reference_states
        from sim.pool8 import active_neurons
        from sim.pickplace import SIGNALS8
        W_RAND = json.loads((DATA / 'week7' / 'week7_random_network_meta.json').read_text())['w_scale']
        rnet = random_network(full, seed=0)
        keep = np.asarray(rnet['keep'])
        eng_r = make_engine(rnet, JO, GPU is not None, GPU_DTYPE)
        enc_r = Encoder(JO, wiring_labels(rnet, JO, 16), SIGNALS8, kind='wiring')
        others = np.setdiff1d(np.arange(len(keep)), np.searchsorted(keep, JO))
        act_l, _ = active_neurons(BatchBrain(eng_r, enc_r, ALL_DN), W_RAND, reference_states(env, EXPERTS[CAND['labels']], OBS_NAMES, n=60), others)
        del eng_r; gc.collect()
        WIDE_R = np.union1d(keep[act_l], ALL_DN)
        NETS['random network'] = run_network9('random network', rnet, JO, WIDE_R, env, OBS_NAMES, CFG, OUT,
                                              w_scale=W_RAND, cand=CAND, resume=RESUME, extra_record=DN_cand)
        del rnet; gc.collect()
    print(f"peak memory {peak_mem_gb():.2f} GB")
else:
    PASS = None
    print("closed loop not run:", WHY)

## Step 3: comparison, video, results file

In [ ]:
from sim.week7 import comparison_rows
SUMM = {k: v['summary'] for k, v in NETS.items()}
SUMM['connectome, wide readout (week 8b)'] = R8B['summary']
W8 = DATA / 'week8' / 'week8_connectome_b.pkl'
if W8.exists():
    SUMM['connectome, DN readout (week 8)'] = pickle.load(open(W8, 'rb'))['summary']
SUMM.update({k: v['summary'] for k, v in REFS.items()})
TABLE = pd.DataFrame(comparison_rows(SUMM)).set_index('policy')
display(TABLE.style.format({c: '{:.0%}' for c in ('success rate', 'lifted', 'carried to zone', 'stalled')}, na_rep='-'))
display(pd.DataFrame({k: s['outcomes'] for k, s in SUMM.items()}).fillna(0).astype(int).T)
VERDICT = None
if 'random network' in NETS:
    from sim.week7 import verdict
    VERDICT, _ = verdict(NETS[LABEL]['outs'], NETS['random network']['outs'])
    print("VERDICT:", VERDICT)
VIDEO = None
if NETS:
    EPISODES = [e for k in NETS for e in NETS[k]['video_eps']]
    for e in EPISODES:
        print(f"{e['net']}, seed {e['seed']}: {e['outcome']}")
    pickle.dump(EPISODES, open(OUT / 'week9_episodes.pkl', 'wb'))
    from sim.video7 import render
    VIDEO = OUT / f'week9_pick_place_{HOST}.mp4'
    if render(str(OUT / 'week9_episodes.pkl'), str(VIDEO), slow=0.5):
        from IPython.display import Video
        display(Video(str(VIDEO), embed=True, width=900))
summary = dict(host=HOST, quick=QUICK, gpu=GPU, k_wide=K_WIDE, rule=RULE, counts=COUNTS,
               offline=OFF.reset_index().to_dict('records'), decision=DECISION, rule_pick=PICK, why=WHY,
               chosen=CHOSEN if RUN else None, forced=FORCE_CANDIDATE, closed_loop=bool(RUN), passed=PASS,
               table=TABLE.reset_index().to_dict('records'), outcomes={k: s['outcomes'] for k, s in SUMM.items()},
               networks={k: dict(dagger_log=v['dagger']['log'], cut=v['cut']['summary'] if 'cut' in v else None,
                                 trap=v['trap'], wall_min=v['wall_min']) for k, v in NETS.items()},
               verdict=VERDICT, wall_min=(time.perf_counter() - T_START) / 60)
(OUT / f'week9_results_{HOST}.json').write_text(json.dumps(summary, indent=1, default=lambda o: o.tolist() if hasattr(o, 'tolist') else str(o)))
print(f"this session {summary['wall_min']:.0f} min. Send me week9_results_{HOST}.json"
      f"{' and the video' if VIDEO else ''} from {OUT}.")